# Module 9 · 文字 1：經典文本表示法（快速回顧）

> **本節定位｜快速帶過（經典基礎）**
> BoW、TF-IDF、靜態詞向量是 2013–2018 的主流，今天仍是理解 NLP 的基礎，
> 但 **2026 大模型資料前處理的主流已換成 subword tokenizer + 上下文嵌入**。
> 本節用一個 notebook 快速複習這三種經典法、點出它們的限制，
> 然後在 `02_tokenization` 正式進入現代管線。

## 學習目標
- 用 sklearn 快速重溫 **詞袋 (BoW)**、**TF-IDF** 兩種稀疏表示法。
- 理解 **靜態詞向量 (Word2Vec / GloVe / spaCy)** 的稠密表示與其「上下文無關」限制。
- 從「資料結構」角度看經典法的輸出：**文件–詞項矩陣 `(N_docs, V)`（稀疏）**。
- 說清楚 **為什麼 2026 要改用 tokenizer + Transformer 上下文嵌入**，銜接下一節。

## 0. 資料結構設計：經典文本表示的輸入與輸出

在動手前，先建立「資料結構」的視角——這是本模組每一節的共同主軸。

| 階段 | 物件 | 形狀 / 型別 | 說明 |
|:--|:--|:--|:--|
| 原始輸入 | `list[str]` | `(N_docs,)` | 一篇文件一個字串 |
| BoW / TF-IDF 輸出 | 稀疏矩陣 `scipy.csr_matrix` | `(N_docs, V)` | V = 詞彙表大小，多為數千~數萬，**高維且稀疏** |
| 靜態詞向量 | `np.ndarray` | 每詞 `(D,)`，D≈50–300 | 一個詞固定一個向量（**不分上下文**） |
| 下游標籤（分類） | `np.ndarray` | `(N_docs,)` | 類別索引 |

對照 2026 現代管線（下一節）：輸入會變成 **`input_ids (B, L)` + `attention_mask (B, L)`** 的整數張量。
形狀的差異，正是「經典 vs 現代」最直觀的分水嶺。

In [1]:
# 經典法只需 scikit-learn（已在核心依賴中），無需下載任何模型。
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 一個迷你語料庫（4 篇短文件）
corpus = [
    "The cat sat on the mat.",
    "A dog sat on the log.",
    "Cats and dogs are common pets.",
    "I love natural language processing.",
]
print(f"文件數 N_docs = {len(corpus)}")

文件數 N_docs = 4


## 1. 詞袋模型 (Bag-of-Words)

BoW 把每篇文件表示成「各詞出現次數」的向量，**忽略詞序**。
輸出是一個 `(N_docs, V)` 的稀疏矩陣。

In [2]:
bow = CountVectorizer()
X_bow = bow.fit_transform(corpus)

print(f"詞彙表大小 V = {len(bow.get_feature_names_out())}")
print(f"BoW 矩陣形狀 = {X_bow.shape}  (N_docs, V)")
print(f"矩陣型別 = {type(X_bow).__name__}（稀疏；非零比例 = {X_bow.nnz / (X_bow.shape[0]*X_bow.shape[1]):.1%}）")
print("\n詞彙表：", list(bow.get_feature_names_out()))
print("\n第 0 篇文件的稠密向量：\n", X_bow.toarray()[0])

詞彙表大小 V = 17
BoW 矩陣形狀 = (4, 17)  (N_docs, V)
矩陣型別 = csr_matrix（稀疏；非零比例 = 29.4%）

詞彙表： ['and', 'are', 'cat', 'cats', 'common', 'dog', 'dogs', 'language', 'log', 'love', 'mat', 'natural', 'on', 'pets', 'processing', 'sat', 'the']

第 0 篇文件的稠密向量：
 [0 0 1 0 0 0 0 0 0 0 1 0 1 0 0 1 2]


**解讀**：向量維度等於詞彙表大小，絕大多數為 0（稀疏）。
「the / cat / sat / mat」在第 0 篇出現，其餘為 0。
BoW 完全不知道 "cat" 和 "cats"、"dog" 和 "dogs" 有關係——這是它的根本限制之一。

## 2. TF-IDF

TF-IDF 在 BoW 基礎上，對「在很多文件都出現的常見詞」降權、對「某文件獨有的詞」升權。
形狀與 BoW 相同 `(N_docs, V)`，只是值從次數換成加權分數。

In [3]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)
print(f"TF-IDF 矩陣形狀 = {X_tfidf.shape}  (N_docs, V)")

# 看第 3 篇「natural language processing」中權重最高的幾個詞
import numpy as np
row = X_tfidf.toarray()[3]
vocab = tfidf.get_feature_names_out()
top = np.argsort(row)[::-1][:5]
print("\n第 3 篇 TF-IDF 權重最高的詞：")
for i in top:
    if row[i] > 0:
        print(f"  {vocab[i]:<12} {row[i]:.3f}")

TF-IDF 矩陣形狀 = (4, 17)  (N_docs, V)

第 3 篇 TF-IDF 權重最高的詞：
  processing   0.500
  natural      0.500
  love         0.500
  language     0.500


**解讀**：像 "natural / language / processing / love" 這類在單篇出現、整體罕見的詞拿到高分；
而 "the / on" 這種到處都有的詞被壓低。TF-IDF 至今仍是**強力的傳統文本分類基線**，
但它和 BoW 一樣是稀疏、上下文無關的詞頻統計。

## 3. 靜態詞向量 (Word2Vec / GloVe / spaCy)

靜態詞向量把每個詞映射到一個低維**稠密**向量（如 300 維），語意相近的詞距離較近
（經典類比：`king - man + woman ≈ queen`）。這是邁向「語意理解」的第一步。

> 執行需要 `classical` extra 與 spaCy 模型：
> `uv sync --extra classical && uv run python -m spacy download en_core_web_md`
> 若環境未安裝，下方程式會跳過並只印出說明（不影響本節重點）。

In [4]:
try:
    import spacy
    nlp = spacy.load("en_core_web_md")
    vec_cat = nlp("cat").vector
    print(f"'cat' 的詞向量形狀 = {vec_cat.shape}  (D,)  ← 稠密、低維")
    print(f"相似度 cat ~ dog  : {nlp('cat').similarity(nlp('dog')):.3f}")
    print(f"相似度 cat ~ car  : {nlp('cat').similarity(nlp('car')):.3f}")
except Exception as e:
    print("（未安裝 spaCy 模型，略過實際計算）")
    print("概念：每個詞 → 一個固定 300 維向量；cat 與 dog 的餘弦相似度會明顯高於 cat 與 car。")

（未安裝 spaCy 模型，略過實際計算）
概念：每個詞 → 一個固定 300 維向量；cat 與 dog 的餘弦相似度會明顯高於 cat 與 car。


**靜態詞向量的致命限制——「一詞一向量」**：

句子 A：*"I deposited money in the **bank**."*（銀行）
句子 B：*"We sat on the river **bank**."*（河岸）

靜態詞向量裡 "bank" **只有一個向量**，無法區分這兩種完全不同的意思。
真實語言充滿一詞多義，這正是 2018 年後 **上下文嵌入 (contextual embeddings)** 興起的原因。

## 4. 為什麼 2026 改用 Tokenizer + Transformer？

| 維度 | 經典法（本節） | 2026 主流（接下來幾節） |
|:--|:--|:--|
| 文本切分 | 以「詞」為單位，遇到生字/拼錯/多語言就爆詞彙表或變 OOV | **Subword tokenizer**（BPE/WordPiece），任何字串都能切，無 OOV |
| 表示形式 | 稀疏 `(N, V)` 或上下文無關的靜態向量 | 整數 `input_ids (B, L)` → Transformer → **隨上下文變化的嵌入** |
| 一詞多義 | ✗ 無法區分 | ✓ "bank" 在不同句子得到不同向量 |
| 下游用途 | 傳統分類器（LogReg/SVM）的特徵 | 微調分類器、LLM、檢索/RAG、生成 |

**結論**：經典法是理解 NLP 的地基，但要餵給 2026 的大模型，
我們需要先學會 **tokenization（下一節）**，再進到 **上下文/句嵌入** 與 **LLM 訓練資料格式**。

---
### 小結
- BoW / TF-IDF：稀疏 `(N_docs, V)`，是強力的傳統基線，但上下文無關。
- 靜態詞向量：稠密、有語意，但「一詞一向量」無法處理多義。
- 這些限制直接導向現代管線 → 下一節 `02_tokenization`。